# Chapter 10: Capstone Projects

**Duration: 3 hours**

## Overview

This chapter presents five capstone projects integrating the skills from Chapters 1--9. Each project is self-contained and can be completed in approximately 30--45 minutes.

## Project 1: Monte Carlo Estimation of Pi

Estimate $\pi$ by randomly throwing darts at a square containing a quarter circle. Uses arrays, broadcasting, and visualization.

In [ ]:
using Plots

function estimate_pi(n)
    x = rand(n)
    y = rand(n)
    inside = x.^2 .+ y.^2 .<= 1.0
    pi_est = 4 * sum(inside) / n
    return pi_est, x, y, inside
end

n = 10_000
pi_est, x, y, inside = estimate_pi(n)
println("π ≈ $pi_est (error: $(abs(pi_est - π)))")

scatter(x[inside], y[inside], markersize=1, label="inside", alpha=0.5)
scatter!(x[.!inside], y[.!inside], markersize=1, label="outside", alpha=0.5)
plot!(title="Monte Carlo π ≈ $(round(pi_est, digits=4))", aspect_ratio=1)

In [ ]:
# Convergence analysis
ns = [10^k for k in 1:6]
estimates = [estimate_pi(n)[1] for n in ns]
errors = abs.(estimates .- π)

plot(ns, errors, xscale=:log10, yscale=:log10,
     xlabel="Number of samples", ylabel="|π_est - π|",
     title="Monte Carlo Convergence", marker=:circle,
     label="MC error", linewidth=2)
plot!(ns, 1.0 ./ sqrt.(ns), linestyle=:dash, label="1/√n", linewidth=2)

## Project 2: Data Analysis Pipeline

Build a complete data analysis pipeline using DataFrames.jl and Plots.jl.

In [ ]:
using DataFrames, Statistics, Plots
using Random
Random.seed!(42)

n = 200
df = DataFrame(
    student_id = 1:n,
    age = rand(18:25, n),
    study_hours = round.(4 .+ 3 .* randn(n), digits=1),
    math_score = rand(40:100, n),
    cs_score = rand(45:100, n),
    major = rand(["Math", "CS", "Physics", "Engineering"], n)
)

# Ensure study_hours is positive
df.study_hours = max.(df.study_hours, 0.5)

# Add a correlated GPA
df.gpa = clamp.(2.0 .+ 0.1 .* df.study_hours .+ 0.01 .* df.math_score .+ 0.3 .* randn(n), 1.0, 4.0)
df.gpa = round.(df.gpa, digits=2)

first(df, 5)

In [ ]:
# Summary statistics by major
combine(groupby(df, :major),
    :gpa => mean => :avg_gpa,
    :study_hours => mean => :avg_study_hours,
    :math_score => mean => :avg_math,
    :cs_score => mean => :avg_cs,
    nrow => :count
)

In [ ]:
# Visualization
p1 = histogram(df.gpa, bins=20, title="GPA Distribution",
               xlabel="GPA", label=false)
p2 = scatter(df.study_hours, df.gpa, xlabel="Study Hours",
             ylabel="GPA", label=false, alpha=0.5,
             title="Study Hours vs GPA")
p3 = histogram(df.math_score, bins=15, title="Math Score Distribution",
               xlabel="Score", label=false)
p4 = scatter(df.math_score, df.cs_score, xlabel="Math Score",
             ylabel="CS Score", label=false, alpha=0.5,
             title="Math vs CS Scores")

plot(p1, p2, p3, p4, layout=(2, 2), size=(900, 700))

## Project 3: Solving a System of ODEs -- SIR Epidemic Model

Model the spread of an infectious disease using the SIR model.

In [ ]:
using OrdinaryDiffEq, Plots

function sir!(du, u, p, t)
    S, I, R = u
    β, γ = p
    N = S + I + R
    du[1] = -β * S * I / N        # dS/dt
    du[2] = β * S * I / N - γ * I  # dI/dt
    du[3] = γ * I                  # dR/dt
end

# Parameters
β = 0.3   # infection rate
γ = 0.1   # recovery rate
R₀ = β / γ  # basic reproduction number
println("R₀ = $R₀")

u0 = [0.99, 0.01, 0.0]  # S(0), I(0), R(0)
tspan = (0.0, 160.0)
p = [β, γ]

prob = ODEProblem(sir!, u0, tspan, p)
sol = solve(prob, Tsit5())

plot(sol, title="SIR Epidemic Model (R₀ = $R₀)",
     xlabel="Time (days)", ylabel="Fraction of Population",
     label=["Susceptible" "Infected" "Recovered"],
     linewidth=2, legend=:right)

In [ ]:
# Parameter study: vary β
plot(title="Effect of β on Infection Peak",
     xlabel="Time", ylabel="Infected fraction")
for β_val in [0.15, 0.2, 0.3, 0.5, 0.8]
    prob = ODEProblem(sir!, u0, tspan, [β_val, γ])
    sol = solve(prob, Tsit5())
    plot!(sol.t, sol[2, :], label="β=$β_val", linewidth=2)
end
plot!()

## Project 4: Image of the Mandelbrot Set

Compute and visualize the Mandelbrot set using arrays and broadcasting.

In [ ]:
using Plots

function mandelbrot(c, max_iter=100)
    z = 0.0 + 0.0im
    for i in 1:max_iter
        z = z^2 + c
        abs(z) > 2.0 && return i
    end
    return max_iter
end

# Generate the grid
nx, ny = 800, 600
xs = range(-2.5, 1.0, length=nx)
ys = range(-1.25, 1.25, length=ny)

# Compute (using comprehension)
M = [mandelbrot(x + y*im) for y in ys, x in xs]

heatmap(xs, ys, M,
    title="Mandelbrot Set",
    xlabel="Re(c)", ylabel="Im(c)",
    color=:inferno, aspect_ratio=1,
    size=(800, 600))

In [ ]:
# Zoom into an interesting region
xs_zoom = range(-0.75, -0.65, length=800)
ys_zoom = range(0.1, 0.2, length=600)
M_zoom = [mandelbrot(x + y*im, 500) for y in ys_zoom, x in xs_zoom]

heatmap(xs_zoom, ys_zoom, M_zoom,
    title="Mandelbrot Set (Zoomed)",
    color=:inferno, aspect_ratio=1)

## Project 5: Build a Simple Neural Network from Scratch

Implement a 2-layer neural network without any ML library to solve XOR.

In [ ]:
using LinearAlgebra, Random
Random.seed!(42)

# Activation functions
sigmoid(x) = 1.0 / (1.0 + exp(-x))
sigmoid_deriv(x) = x * (1.0 - x)

# Generate XOR dataset
X = Float64[0 0; 0 1; 1 0; 1 1]'   # 2×4
y = Float64[0 1 1 0]                # 1×4 (reshaped below)
Y = reshape(y, 1, 4)

# Initialize weights
W1 = randn(4, 2) * 0.5   # hidden layer: 4 neurons, 2 inputs
b1 = zeros(4, 1)
W2 = randn(1, 4) * 0.5   # output layer: 1 neuron, 4 inputs
b2 = zeros(1, 1)

lr = 1.0
losses = Float64[]

for epoch in 1:5000
    # Forward pass
    Z1 = W1 * X .+ b1
    A1 = sigmoid.(Z1)
    Z2 = W2 * A1 .+ b2
    A2 = sigmoid.(Z2)

    # Loss
    loss = sum((A2 .- Y).^2) / 4
    push!(losses, loss)

    # Backward pass
    dA2 = (A2 .- Y) .* sigmoid_deriv.(A2)
    dW2 = dA2 * A1' / 4
    db2 = sum(dA2, dims=2) / 4

    dA1 = (W2' * dA2) .* sigmoid_deriv.(A1)
    dW1 = dA1 * X' / 4
    db1 = sum(dA1, dims=2) / 4

    # Update
    W2 .-= lr .* dW2
    b2 .-= lr .* db2
    W1 .-= lr .* dW1
    b1 .-= lr .* db1
end

println("Final loss: $(losses[end])")
println("Predictions: $(round.(sigmoid.(W2 * sigmoid.(W1 * X .+ b1) .+ b2), digits=3))")
println("Expected:    [0, 1, 1, 0]")

In [ ]:
# Plot training loss
using Plots
plot(losses, yscale=:log10,
     title="XOR Neural Network Training",
     xlabel="Epoch", ylabel="MSE Loss",
     label="loss", linewidth=2)

## Capstone Exercises

In [ ]:
# Capstone Exercise 1: Extend the Monte Carlo project to estimate
# the volume of a 10-dimensional unit hypersphere.
# Compare with the analytical formula.
# Your code here

In [ ]:
# Capstone Exercise 2: Build a complete ML pipeline using MLJ:
# load data → clean → feature engineering → train → evaluate → visualize.
# Your code here

In [ ]:
# Capstone Exercise 3: Implement the Lorenz attractor and create
# an animation showing the trajectory evolving over time.
# Your code here

In [ ]:
# Capstone Exercise 4: Solve the heat equation (PDE) using
# finite differences. Visualize the solution over time.
# Your code here

In [ ]:
# Capstone Exercise 5: Download a real dataset (e.g., from UCI ML Repository),
# perform EDA with DataFrames, train multiple models with MLJ,
# and compare their performance.
# Your code here